# Colab 8 — Siamese Network + k-NN for Sequence Similarity

Pivot from the DP-mirror FNN (colabs 1–7) to an **embedding-based approach**:  
a Siamese network maps sequences to a shared vector space where L2 distance approximates normalized Levenshtein distance.  
This enables k-NN retrieval of similar sequences without pairwise DP computation.

**Prototype universe:** all 32 binary strings of length 5 (same as colabs 6–7).

```
              Shared Encoder
  seq_A ──► Embed → Conv1D → Conv1D → AvgPool → Linear → L2Norm ──► emb_A
  seq_B ──► (same weights)                                       ──► emb_B

  predicted_distance = ‖emb_A − emb_B‖₂
  target             = lev(A, B) / 5
```

## 1. Setup & Imports

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Data Generation

In [ ]:
def levenshtein(a, b):
    """Standard Wagner-Fischer DP for Levenshtein distance."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

SEQ_LEN = 5
VOCAB_SIZE = 2  # binary: {0, 1}

# Generate all 32 binary strings of length 5
all_strings = [''.join(str(b) for b in bits) for bits in product(range(VOCAB_SIZE), repeat=SEQ_LEN)]
print(f'Number of strings: {len(all_strings)}')
print(f'Examples: {all_strings[:5]} ... {all_strings[-3:]}')

# Compute all 1024 ordered pairs (32 x 32) with distances
pairs = []
for a in all_strings:
    for b in all_strings:
        d = levenshtein(a, b)
        pairs.append((a, b, d / SEQ_LEN))  # normalized distance

print(f'Total pairs: {len(pairs)}')
print(f'Distance distribution (normalized):')
from collections import Counter
dist_counts = Counter(round(p[2], 2) for p in pairs)
for d in sorted(dist_counts.keys()):
    print(f'  {d:.2f}: {dist_counts[d]} pairs')

## 3. PyTorch Dataset & DataLoader

In [ ]:
class SequencePairDataset(Dataset):
    def __init__(self, pairs, oversample_threshold=0.4, oversample_factor=3):
        self.data = []
        for seq_a, seq_b, norm_dist in pairs:
            entry = (
                torch.tensor([int(c) for c in seq_a], dtype=torch.long),
                torch.tensor([int(c) for c in seq_b], dtype=torch.long),
                torch.tensor(norm_dist, dtype=torch.float32)
            )
            self.data.append(entry)
            # Oversample close pairs
            if norm_dist < oversample_threshold:
                for _ in range(oversample_factor - 1):
                    self.data.append(entry)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

dataset = SequencePairDataset(pairs)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f'Dataset size (with oversampling): {len(dataset)}')
print(f'Original pairs: {len(pairs)}')
print(f'Oversampled close pairs added: {len(dataset) - len(pairs)}')

## 4. Siamese Encoder Model

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=2, embed_dim=16, hidden1=32, hidden2=64, out_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Conv1d expects (batch, channels, length)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2, out_dim)

    def forward(self, x):
        # x: (batch, seq_len) of token indices
        x = self.embedding(x)          # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)         # (batch, embed_dim, seq_len)
        x = F.relu(self.conv1(x))       # (batch, hidden1, seq_len)
        x = F.relu(self.conv2(x))       # (batch, hidden2, seq_len)
        x = x.mean(dim=2)               # Global average pool → (batch, hidden2)
        x = self.fc(x)                  # (batch, out_dim)
        x = F.normalize(x, p=2, dim=1)  # L2 normalize
        return x


class SiameseNetwork(nn.Module):
    def __init__(self, vocab_size=2, embed_dim=16, out_dim=128):
        super().__init__()
        self.encoder = SiameseEncoder(vocab_size, embed_dim, out_dim=out_dim)

    def forward(self, seq_a, seq_b):
        emb_a = self.encoder(seq_a)
        emb_b = self.encoder(seq_b)
        # L2 distance between embeddings
        dist = torch.norm(emb_a - emb_b, p=2, dim=1)
        return dist

    def encode(self, seq):
        return self.encoder(seq)


model = SiameseNetwork(vocab_size=VOCAB_SIZE).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 5. Weighted MSE Loss

Weight = exp(-α · true_dist) with α=3.  
This prioritizes getting close-pair distances right — weight is ~1.0 at dist=0, ~0.30 at dist=0.4, ~0.05 at dist=1.0.

In [ ]:
def weighted_mse_loss(pred_dist, true_dist, alpha=3.0):
    weight = torch.exp(-alpha * true_dist)
    return torch.mean(weight * (pred_dist - true_dist) ** 2)

# Visualize the weight function
d_vals = np.linspace(0, 1, 100)
w_vals = np.exp(-3.0 * d_vals)
plt.figure(figsize=(6, 3))
plt.plot(d_vals, w_vals)
plt.axvline(x=0.4, color='r', linestyle='--', label='threshold = 0.4')
plt.xlabel('Normalized Levenshtein distance')
plt.ylabel('Weight')
plt.title('Weighted MSE: weight = exp(-3 · d)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 200
losses = []

model.train()
for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    num_batches = 0
    for seq_a, seq_b, true_dist in dataloader:
        seq_a = seq_a.to(device)
        seq_b = seq_b.to(device)
        true_dist = true_dist.to(device)

        pred_dist = model(seq_a, seq_b)
        loss = weighted_mse_loss(pred_dist, true_dist)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{num_epochs} — Loss: {avg_loss:.6f}')

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Weighted MSE Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final loss: {losses[-1]:.6f}')

## 7. Embedding Visualization

Encode all 32 sequences, project to 2D with PCA and t-SNE.  
Color = Levenshtein distance to reference string `'00000'`.

In [ ]:
model.eval()

# Encode all 32 sequences
all_tensors = torch.stack([
    torch.tensor([int(c) for c in s], dtype=torch.long) for s in all_strings
]).to(device)

with torch.no_grad():
    all_embeddings = model.encode(all_tensors).cpu().numpy()

print(f'Embedding matrix shape: {all_embeddings.shape}')  # (32, 128)

# Distance to reference string '00000'
ref = '00000'
colors = [levenshtein(s, ref) for s in all_strings]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA
pca_2d = PCA(n_components=2).fit_transform(all_embeddings)
sc1 = axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=colors, cmap='viridis', s=60, edgecolors='k', linewidth=0.5)
for i, s in enumerate(all_strings):
    axes[0].annotate(s, (pca_2d[i, 0], pca_2d[i, 1]), fontsize=5, alpha=0.7)
axes[0].set_title('PCA projection')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# t-SNE
tsne_2d = TSNE(n_components=2, perplexity=10, random_state=SEED).fit_transform(all_embeddings)
sc2 = axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=colors, cmap='viridis', s=60, edgecolors='k', linewidth=0.5)
for i, s in enumerate(all_strings):
    axes[1].annotate(s, (tsne_2d[i, 0], tsne_2d[i, 1]), fontsize=5, alpha=0.7)
axes[1].set_title('t-SNE projection')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

fig.colorbar(sc2, ax=axes, label=f'Levenshtein distance to \'{ref}\'')
plt.suptitle('Embedding Space — 32 Binary Strings of Length 5', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Distance Correlation Plot

For all 496 unique pairs: true normalized Levenshtein distance vs. predicted embedding L2 distance.

In [ ]:
# Compute all pairwise embedding distances
emb_tensor = torch.tensor(all_embeddings, dtype=torch.float32)
# Pairwise L2 distance matrix
emb_dists = torch.cdist(emb_tensor, emb_tensor, p=2).numpy()

# Collect unique pairs (i < j)
true_dists_unique = []
pred_dists_unique = []
n = len(all_strings)
for i in range(n):
    for j in range(i + 1, n):
        true_dists_unique.append(levenshtein(all_strings[i], all_strings[j]) / SEQ_LEN)
        pred_dists_unique.append(emb_dists[i, j])

true_dists_unique = np.array(true_dists_unique)
pred_dists_unique = np.array(pred_dists_unique)

# Correlations
pearson_r, pearson_p = pearsonr(true_dists_unique, pred_dists_unique)
spearman_r, spearman_p = spearmanr(true_dists_unique, pred_dists_unique)
print(f'Pearson  r = {pearson_r:.4f} (p = {pearson_p:.2e})')
print(f'Spearman ρ = {spearman_r:.4f} (p = {spearman_p:.2e})')

# Scatter plot
plt.figure(figsize=(7, 6))
plt.scatter(true_dists_unique, pred_dists_unique, alpha=0.3, s=15)

# Linear regression line
coeffs = np.polyfit(true_dists_unique, pred_dists_unique, 1)
x_line = np.linspace(0, 1, 100)
plt.plot(x_line, np.polyval(coeffs, x_line), 'r-', linewidth=2,
         label=f'Linear fit (slope={coeffs[0]:.3f})')

plt.xlabel('True Normalized Levenshtein Distance')
plt.ylabel('Predicted Embedding L2 Distance')
plt.title(f'Distance Correlation\nPearson r={pearson_r:.3f}, Spearman ρ={spearman_r:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Sanity checks
idx_00000 = all_strings.index('00000')
idx_11111 = all_strings.index('11111')
print(f'\nSanity checks:')
print(f'  Distance "00000" to itself: {emb_dists[idx_00000, idx_00000]:.6f} (should be ~0)')
print(f'  Distance "00000" to "11111": {emb_dists[idx_00000, idx_11111]:.6f} (should be large)')
print(f'  Max embedding distance: {emb_dists.max():.6f}')

## 9. k-NN Evaluation

For each string, compare the true k nearest neighbors (by Levenshtein) with the predicted k nearest neighbors (by embedding L2 distance).  
Metric: **Recall@k** = |true_kNN ∩ predicted_kNN| / k

In [ ]:
# Precompute true Levenshtein distance matrix
n = len(all_strings)
true_dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        true_dist_matrix[i, j] = levenshtein(all_strings[i], all_strings[j])

def get_knn(dist_matrix, query_idx, k):
    """Return indices of k nearest neighbors (excluding self)."""
    dists = dist_matrix[query_idx].copy()
    dists[query_idx] = np.inf  # exclude self
    return set(np.argsort(dists)[:k])

# Evaluate Recall@k for k = 1, 3, 5
for k in [1, 3, 5]:
    recalls = []
    for i in range(n):
        true_knn = get_knn(true_dist_matrix, i, k)
        pred_knn = get_knn(emb_dists, i, k)
        recall = len(true_knn & pred_knn) / k
        recalls.append(recall)
    mean_recall = np.mean(recalls)
    print(f'Mean Recall@{k}: {mean_recall:.4f}')

print()

# Show examples for a few query strings
example_queries = ['00000', '01010', '11111', '00110']
k_example = 5

print(f'Example k-NN queries (k={k_example}):')
print('=' * 70)
for query in example_queries:
    qi = all_strings.index(query)
    true_knn_idx = get_knn(true_dist_matrix, qi, k_example)
    pred_knn_idx = get_knn(emb_dists, qi, k_example)
    overlap = true_knn_idx & pred_knn_idx

    true_neighbors = [(all_strings[j], int(true_dist_matrix[qi, j])) for j in sorted(true_knn_idx)]
    pred_neighbors = [(all_strings[j], f'{emb_dists[qi, j]:.4f}') for j in sorted(pred_knn_idx)]

    print(f'\nQuery: \'{query}\'')
    print(f'  True  top-{k_example} (string, lev_dist): {true_neighbors}')
    print(f'  Pred  top-{k_example} (string, emb_dist): {pred_neighbors}')
    print(f'  Recall@{k_example}: {len(overlap)}/{k_example} = {len(overlap)/k_example:.2f}')